# NOTEBOOK 04 — Build global aggs and null features

Цель:
- добавить независимый блок сильных признаков поверх `top-700 extra`;
- построить row-level global aggregations;
- построить `Null Pattern SVD` по бинарной матрице пропусков.

Что сохраняем:
- `artifacts/aggs/train_global_aggs.parquet`
- `artifacts/aggs/test_global_aggs.parquet`
- `artifacts/aggs/train_null_svd.parquet`
- `artifacts/aggs/test_null_svd.parquet`

Логика:
- upstream-артефакты читаются как Kaggle input dataset из `NOTEBOOK 01`;
- новые файлы пишутся только в `kaggle/working` текущей сессии;
- train и test обрабатываются отдельно для бережной памяти.


In [6]:
# =========================
# Imports
# =========================
from pathlib import Path
import gc
import json
import warnings

import numpy as np
import polars as pl
from sklearn.decomposition import TruncatedSVD
from scipy import sparse

warnings.filterwarnings("ignore")

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)


polars.config.Config

In [7]:
# =========================
# Paths
# =========================
DATA_DIR = Path("/kaggle/input/datasets/hatab123/data-fusion-contest-2026/")

# output dataset from NOTEBOOK 01
NB01_DIR = Path("/kaggle/input/notebooks/viktoriasvetankova/01-feature-selection-41targets2803/prepared")

# current-session working dir
WORK_DIR = Path("/kaggle/working/prepared")
WORK_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "train_extra": DATA_DIR / "train_extra_features.parquet",
    "test_extra": DATA_DIR / "test_extra_features.parquet",
}

UPSTREAM_PATHS = {
    "selected_top700": NB01_DIR / "artifacts" / "selected_features" / "selected_extra_top700.json",
}

AGGS_DIR = WORK_DIR / "artifacts" / "aggs"
LOG_DIR = WORK_DIR / "artifacts" / "logs"

AGGS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("NB01_DIR:", NB01_DIR)
print("WORK_DIR:", WORK_DIR)
for name, path in PATHS.items():
    print(f"{name:>12}: {path} | exists={path.exists()}")
for name, path in UPSTREAM_PATHS.items():
    print(f"{name:>12}: {path} | exists={path.exists()}")


DATA_DIR: /kaggle/input/datasets/hatab123/data-fusion-contest-2026
NB01_DIR: /kaggle/input/notebooks/viktoriasvetankova/01-feature-selection-41targets2803/prepared
WORK_DIR: /kaggle/working/prepared
 train_extra: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_extra_features.parquet | exists=True
  test_extra: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/test_extra_features.parquet | exists=True
selected_top700: /kaggle/input/notebooks/viktoriasvetankova/01-feature-selection-41targets2803/prepared/artifacts/selected_features/selected_extra_top700.json | exists=True


In [8]:
# =========================
# Config
# =========================
ID_COL = "customer_id"
TOP700_K = 700
NULL_SVD_COMPONENTS = 20
FLOAT_DTYPE = np.float32

TRAIN_AGGS_OUT = AGGS_DIR / "train_global_aggs.parquet"
TEST_AGGS_OUT = AGGS_DIR / "test_global_aggs.parquet"
TRAIN_SVD_OUT = AGGS_DIR / "train_null_svd.parquet"
TEST_SVD_OUT = AGGS_DIR / "test_null_svd.parquet"
NULL_SVD_META_OUT = LOG_DIR / "null_svd_meta.json"


In [9]:
# =========================
# Helpers
# =========================
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path: Path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def report_mem(df: pl.DataFrame, name: str):
    mem_mb = df.estimated_size() / 1024**2
    print(f"{name}: shape={df.shape}, memory={mem_mb:,.2f} MB")

def downcast_pl(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for col, dtype in zip(df.columns, df.dtypes):
        if dtype == pl.Float64:
            exprs.append(pl.col(col).cast(pl.Float32))
        elif dtype == pl.Int64:
            exprs.append(pl.col(col).cast(pl.Int32))
        elif dtype == pl.UInt64:
            exprs.append(pl.col(col).cast(pl.UInt32))
        else:
            exprs.append(pl.col(col))
    return df.select(exprs)

def load_extra_block(path: Path, feature_cols):
    cols = [ID_COL] + feature_cols
    df = pl.read_parquet(path, columns=cols)
    df = downcast_pl(df)
    return df

def build_global_aggs(df_extra: pl.DataFrame, feature_cols):
    X = df_extra.select(feature_cols).to_numpy()
    if X.dtype != np.float32:
        X = X.astype(np.float32, copy=False)

    null_mask = np.isnan(X)
    notnull_mask = ~null_mask

    # use nan-aware reducers
    sum_extra = np.nansum(X, axis=1).astype(np.float32)
    mean_extra = np.nanmean(X, axis=1).astype(np.float32)
    std_extra = np.nanstd(X, axis=1).astype(np.float32)
    min_extra = np.nanmin(X, axis=1).astype(np.float32)
    max_extra = np.nanmax(X, axis=1).astype(np.float32)

    # counts
    count_nonzero_extra = np.count_nonzero(np.where(null_mask, 0.0, X), axis=1).astype(np.int32)
    count_null_extra = null_mask.sum(axis=1).astype(np.int32)

    out = pl.DataFrame({
        ID_COL: df_extra.get_column(ID_COL),
        "sum_extra": sum_extra,
        "mean_extra": mean_extra,
        "std_extra": std_extra,
        "min_extra": min_extra,
        "max_extra": max_extra,
        "count_nonzero_extra": count_nonzero_extra,
        "count_null_extra": count_null_extra,
    })

    del X, null_mask, notnull_mask, sum_extra, mean_extra, std_extra, min_extra, max_extra
    del count_nonzero_extra, count_null_extra
    gc.collect()

    return out

def build_null_mask_csr(df_extra: pl.DataFrame, feature_cols):
    X = df_extra.select(feature_cols).to_numpy()
    if X.dtype != np.float32:
        X = X.astype(np.float32, copy=False)

    null_mask = np.isnan(X).astype(np.float32, copy=False)
    X_sparse = sparse.csr_matrix(null_mask)

    del X, null_mask
    gc.collect()

    return X_sparse

def make_null_svd_df(ids, arr, prefix="null_svd"):
    cols = {ID_COL: ids}
    for i in range(arr.shape[1]):
        cols[f"{prefix}_{i:02d}"] = arr[:, i].astype(np.float32, copy=False)
    return pl.DataFrame(cols)

def sanitize_nan_aggs(df: pl.DataFrame):
    # just in case an all-null row appears for reducers like nanmean/nanstd/nanmin/nanmax
    fill_exprs = []
    for col, dtype in zip(df.columns, df.dtypes):
        if col == ID_COL:
            fill_exprs.append(pl.col(col))
        elif dtype in (pl.Float32, pl.Float64):
            fill_exprs.append(pl.col(col).fill_nan(0.0).fill_null(0.0).alias(col))
        else:
            fill_exprs.append(pl.col(col).fill_null(0).alias(col))
    return df.select(fill_exprs)


In [10]:
# =========================
# Load selected top-700 extra features
# =========================
extra_top700 = load_json(UPSTREAM_PATHS["selected_top700"])
assert isinstance(extra_top700, list), "selected_extra_top700.json must contain a list"
assert len(extra_top700) > 0, "top700 list is empty"

print("extra_top700:", len(extra_top700))
print(extra_top700[:10], "...")


extra_top700: 700
['num_feature_879', 'num_feature_1390', 'num_feature_1370', 'num_feature_1459', 'num_feature_1649', 'num_feature_376', 'num_feature_1672', 'num_feature_1355', 'num_feature_841', 'num_feature_2348'] ...


## 1. Train extra block → global aggs + Null SVD fit

In [11]:
# =========================
# Train block
# =========================
train_extra = load_extra_block(PATHS["train_extra"], extra_top700)
report_mem(train_extra, "train_extra[top700]")

train_global_aggs = build_global_aggs(train_extra, extra_top700)
train_global_aggs = sanitize_nan_aggs(train_global_aggs)
report_mem(train_global_aggs, "train_global_aggs")

# Null Pattern SVD on top-700 extra (fit on train+test null matrix)
train_null_sparse = build_null_mask_csr(train_extra, extra_top700)
print("train_null_sparse:", train_null_sparse.shape, "| nnz:", train_null_sparse.nnz)

test_extra_for_svd = load_extra_block(PATHS["test_extra"], extra_top700)
report_mem(test_extra_for_svd, "test_extra_for_svd[top700]")
test_null_sparse_for_fit = build_null_mask_csr(test_extra_for_svd, extra_top700)
print("test_null_sparse_for_fit:", test_null_sparse_for_fit.shape, "| nnz:", test_null_sparse_for_fit.nnz)

full_null_sparse = sparse.vstack([train_null_sparse, test_null_sparse_for_fit], format="csr")
print("full_null_sparse:", full_null_sparse.shape, "| nnz:", full_null_sparse.nnz)

svd = TruncatedSVD(
    n_components=NULL_SVD_COMPONENTS,
    algorithm="randomized",
    n_iter=7,
    random_state=42,
)

svd.fit(full_null_sparse)
train_null_svd_arr = svd.transform(train_null_sparse).astype(np.float32, copy=False)
train_null_svd = make_null_svd_df(
    ids=train_extra.get_column(ID_COL),
    arr=train_null_svd_arr,
    prefix="null_svd",
)
report_mem(train_null_svd, "train_null_svd")

# save train outputs early
train_global_aggs.write_parquet(TRAIN_AGGS_OUT)
train_null_svd.write_parquet(TRAIN_SVD_OUT)

print("Saved:", TRAIN_AGGS_OUT)
print("Saved:", TRAIN_SVD_OUT)

svd_meta = {
    "n_components": NULL_SVD_COMPONENTS,
    "explained_variance_ratio_sum": float(np.sum(svd.explained_variance_ratio_)),
    "explained_variance_ratio": [float(x) for x in svd.explained_variance_ratio_],
    "top700_count": len(extra_top700),
}
save_json(NULL_SVD_META_OUT, svd_meta)
print("Saved:", NULL_SVD_META_OUT)

# free train heavy objects before test
del full_null_sparse, test_null_sparse_for_fit
del test_extra_for_svd
del train_null_sparse, train_null_svd_arr
del train_global_aggs, train_null_svd
del train_extra
gc.collect()


train_extra[top700]: shape=(750000, 701), memory=2,068.16 MB
train_global_aggs: shape=(750000, 8), memory=22.89 MB
train_null_sparse: (750000, 700) | nnz: 296301223
test_extra_for_svd[top700]: shape=(250000, 701), memory=689.39 MB
test_null_sparse_for_fit: (250000, 700) | nnz: 98771107
full_null_sparse: (1000000, 700) | nnz: 395072330
train_null_svd: shape=(750000, 21), memory=60.08 MB
Saved: /kaggle/working/prepared/artifacts/aggs/train_global_aggs.parquet
Saved: /kaggle/working/prepared/artifacts/aggs/train_null_svd.parquet
Saved: /kaggle/working/prepared/artifacts/logs/null_svd_meta.json


33

null_svd
- раньше (в старом ноутбуке) компоненты искались только по train missing-patterns
- теперь компоненты ищутся по общей missing-структуре train+test

## 2. Test extra block → global aggs + Null SVD transform

In [12]:
# =========================
# Test block
# =========================
test_extra = load_extra_block(PATHS["test_extra"], extra_top700)
report_mem(test_extra, "test_extra[top700]")

test_global_aggs = build_global_aggs(test_extra, extra_top700)
test_global_aggs = sanitize_nan_aggs(test_global_aggs)
report_mem(test_global_aggs, "test_global_aggs")

test_null_sparse = build_null_mask_csr(test_extra, extra_top700)
# nnz = число единиц в sparse matrix, то есть сколько всего пропусков отмечено.
print("test_null_sparse:", test_null_sparse.shape, "| nnz:", test_null_sparse.nnz)

test_null_svd_arr = svd.transform(test_null_sparse).astype(np.float32, copy=False)
test_null_svd = make_null_svd_df(
    ids=test_extra.get_column(ID_COL),
    arr=test_null_svd_arr,
    prefix="null_svd",
)
report_mem(test_null_svd, "test_null_svd")

test_global_aggs.write_parquet(TEST_AGGS_OUT)
test_null_svd.write_parquet(TEST_SVD_OUT)

print("Saved:", TEST_AGGS_OUT)
print("Saved:", TEST_SVD_OUT)

del test_null_sparse, test_null_svd_arr
del test_global_aggs, test_null_svd
del test_extra
gc.collect()


test_extra[top700]: shape=(250000, 701), memory=689.39 MB
test_global_aggs: shape=(250000, 8), memory=7.63 MB
test_null_sparse: (250000, 700) | nnz: 98771107
test_null_svd: shape=(250000, 21), memory=20.03 MB
Saved: /kaggle/working/prepared/artifacts/aggs/test_global_aggs.parquet
Saved: /kaggle/working/prepared/artifacts/aggs/test_null_svd.parquet


0

In [13]:
# =========================
# Quick sanity checks
# =========================
train_global_aggs_check = pl.read_parquet(TRAIN_AGGS_OUT)
test_global_aggs_check = pl.read_parquet(TEST_AGGS_OUT)
train_null_svd_check = pl.read_parquet(TRAIN_SVD_OUT)
test_null_svd_check = pl.read_parquet(TEST_SVD_OUT)

report_mem(train_global_aggs_check, "train_global_aggs_check")
report_mem(test_global_aggs_check, "test_global_aggs_check")
report_mem(train_null_svd_check, "train_null_svd_check")
report_mem(test_null_svd_check, "test_null_svd_check")

print("train_global_aggs columns:", train_global_aggs_check.columns)
print("train_null_svd columns   :", train_null_svd_check.columns[:5], "...", train_null_svd_check.columns[-3:])

display(train_global_aggs_check.head(3))
display(train_null_svd_check.head(3))

train_global_aggs_check: shape=(750000, 8), memory=22.89 MB
test_global_aggs_check: shape=(250000, 8), memory=7.63 MB
train_null_svd_check: shape=(750000, 21), memory=60.08 MB
test_null_svd_check: shape=(250000, 21), memory=20.03 MB
train_global_aggs columns: ['customer_id', 'sum_extra', 'mean_extra', 'std_extra', 'min_extra', 'max_extra', 'count_nonzero_extra', 'count_null_extra']
train_null_svd columns   : ['customer_id', 'null_svd_00', 'null_svd_01', 'null_svd_02', 'null_svd_03'] ... ['null_svd_17', 'null_svd_18', 'null_svd_19']


customer_id,sum_extra,mean_extra,std_extra,min_extra,max_extra,count_nonzero_extra,count_null_extra
i32,f32,f32,f32,f32,f32,i32,i32
1000001,-19.372976,-0.063727,0.546359,-0.919307,5.265816,304,396
1000002,23.02689,0.056438,0.622135,-1.785514,5.069804,408,292
1000003,-18.791224,-0.055926,0.967207,-1.659281,11.58178,336,364


customer_id,null_svd_00,null_svd_01,null_svd_02,null_svd_03,null_svd_04,null_svd_05,null_svd_06,null_svd_07,null_svd_08,…,null_svd_10,null_svd_11,null_svd_12,null_svd_13,null_svd_14,null_svd_15,null_svd_16,null_svd_17,null_svd_18,null_svd_19
i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
1000001,17.807964,-3.27305,-0.795936,-1.215483,-1.295941,4.701485,0.481,3.798483,-0.476303,…,2.036204,-0.276354,0.804336,-0.031815,-0.81338,-0.361066,1.092495,0.130342,0.650639,-0.641832
1000002,14.682601,-5.22077,-3.648647,0.080012,-0.515638,-1.722188,-0.909207,0.408838,-1.135478,…,-0.255419,0.18218,0.351931,0.833397,0.510661,1.567058,0.331607,-0.96935,0.060767,-0.925299
1000003,17.277189,-4.587068,-1.839801,-1.005513,-0.378881,-2.592289,1.897362,1.906662,1.295119,…,0.129353,1.912827,1.465122,0.804774,-0.577758,-0.277785,-0.336493,0.657997,1.543059,-0.144223


In [14]:
# =========================
# Final cleanup
# =========================
del train_global_aggs_check, test_global_aggs_check
del train_null_svd_check, test_null_svd_check
del svd
gc.collect()

print("Notebook 04 finished successfully.")

Notebook 04 finished successfully.
